In [34]:
import os
import time
import json
import numpy as np
import pandas as pd
import pm4py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import optuna
import warnings
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding, Concatenate
from tensorflow.keras.optimizers import Adam
from ACF_code import activity_contex_frequency, pmi


In [40]:
# Importing dataset from file path
def import_xes(file_path):
    log = pm4py.read_xes(file_path)
    return pm4py.convert_to_dataframe(log)

# Cleaning dataset: removing unnecessary columns, shifting to resource focus
def clean_dataset(df):
    df_final = df[['case:concept:name', 'concept:name', 'org:resource', 'time:timestamp']]
    df_final = df_final.sort_values(['org:resource', 'time:timestamp'])
    return df_final


# creating 80/20 split based on resources, ensuring a resource is in EITHER the test set OR the train set
def create_split(df_clean, test_size):
    resource_traces = (
        df_clean.sort_values(["org:resource", "time:timestamp"])
               .groupby("org:resource")["concept:name"]
               .apply(list)
    )

    resource_traces = resource_traces[resource_traces.apply(len) > 1]

    resources = resource_traces.index.tolist()

    # create set of train/test resource ids
    train_res, test_res = train_test_split(
        resources,
        test_size=test_size,
        random_state=1
    )

    train_traces = resource_traces.loc[train_res]
    test_traces = resource_traces.loc[test_res]

    return train_traces, test_traces


# prefix extraction on set list of prefix lengths, already implicitly buckets on prefix length
def build_prefix_df(resource_traces, prefix_lengths=[10], sliding_window=False):
    all_rows = []
    for length in prefix_lengths:
        for resource, seq in resource_traces.items():

            if(len(seq) < length + 1):
                continue

            if(sliding_window):
                for i in range(length, len(seq)):
                    prefix = seq[i-length:i]
                    next_act = seq[i]

                    all_rows.append({
                    'resource': resource,
                    'subtrace': prefix,
                    'prefix_length': length,
                    'last_activity': prefix[-1],
                    'next_activity': next_act
                    })
            else:
                prefix = seq[-(length+1):-1]
                next_act = seq[-1]

                all_rows.append({
                'resource': resource,
                'subtrace': prefix,
                'prefix_length': length,
                'last_activity': prefix[-1],
                'next_activity': next_act
                })
            
    return pd.DataFrame(all_rows)




def process_dataset(df, prefix_lengths):
    df_clean = clean_dataset(df)

    train_split, test_split = create_split(df_clean, 0.2)

    train_df = build_prefix_df(train_split, prefix_lengths, True)
    test_df = build_prefix_df(test_split, prefix_lengths, True)

    return train_df, test_df, train_split, test_split


In [4]:
# df_2013, df_2013_prefixes = process_dataset("datasets/BPI_Challenge_2013_incidents.xes")
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")
#train_df_2013, test_df_2013 = process_dataset(df_2013)
GLOBAL_prefix_lengths = [10, 20, 30, 40, 50, 75, 100, 125, 150]
#GLOBAL_prefix_lengths = [100, 150, 200, 300, 400, 500, 600, 700, 800]

train_df_2013, test_df_2013 = process_dataset(df_2013, GLOBAL_prefix_lengths)

/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/pm4py/utils.py:992: UserWarning: Install the optional requirement `rustxes` to import/export files faster.
  warnings.warn("Install the optional requirement `rustxes` to import/export files faster.")
parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 3319.05it/s]


In [36]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

class LSTMDataProcessor:
    def __init__(self):
        self.tokenizer = Tokenizer(filters='', lower=False)
        self.max_len = 0
        self.vocab_size = 0
        self.classes = None

    def fit(self, train_df, test_df):
        """
        Fits the tokenizer on the combined corpus of activities to ensure
        consistent mapping between train and test sets.
        """
        # Gather all unique activities from subtraces and next_activities
        all_traces = list(train_df['subtrace']) + list(test_df['subtrace'])
        all_next = list(train_df['next_activity']) + list(test_df['next_activity'])
        
        # We need to flatten the lists of subtraces for the tokenizer
        corpus = [act for sublist in all_traces for act in sublist] + all_next
        
        self.tokenizer.fit_on_texts(corpus)
        
        # +1 for zero padding
        self.vocab_size = len(self.tokenizer.word_index) + 1
        self.classes = sorted(list(self.tokenizer.word_index.keys()))
        
        # Calculate max length for padding (based on your longest prefix)
        max_train = train_df['subtrace'].apply(len).max()
        max_test = test_df['subtrace'].apply(len).max()
        self.max_len = max(max_train, max_test)
        
        print(f"Vocab Size: {self.vocab_size}")
        print(f"Max Sequence Length: {self.max_len}")

    def transform(self, df):
        """
        Converts the dataframe into X (padded sequences) and y (one-hot targets)
        """
        # 1. Convert lists of strings to lists of integers
        X_seq = self.tokenizer.texts_to_sequences(df['subtrace'])
        
        # 2. Pad sequences to ensure uniform length for the LSTM
        # 'pre' padding is usually better for LSTMs as the last steps are most relevant
        X_padded = pad_sequences(X_seq, maxlen=self.max_len, padding='pre')
        
        # 3. Handle Targets
        y_seq = self.tokenizer.texts_to_sequences(df['next_activity'])
        # Flatten list of lists [[1], [4]] -> [1, 4]
        y_flat = [item[0] for item in y_seq]
        
        # One-hot encode targets
        # Note: to_categorical creates a matrix of size (N, vocab_size + 1)
        # We slice [:, 1:] if we want to ignore the 0 index, but usually keeping it simple is safer
        y_encoded = to_categorical(y_flat, num_classes=self.vocab_size)
        
        return X_padded, y_encoded, y_flat

In [37]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Embedding, Dropout, Input

def build_tax_model(vocab_size, input_length, embedding_dim=50, embedding_type='learned'):
    model = Sequential()
    
    # --- 1. Embedding / Encoding Layer ---
    if embedding_type == 'one_hot':
        # Simulating one-hot by using an embedding layer where weights are fixed 
        # to the identity matrix. This is more efficient than passing sparse matrices.
        # Output dim = Vocab size
        model.add(Embedding(input_dim=vocab_size, 
                            output_dim=vocab_size, 
                            input_length=input_length,
                            embeddings_initializer='identity',
                            trainable=False)) # Fix weights so they act as simple lookup
                            
    else: # 'learned' embeddings
        model.add(Embedding(input_dim=vocab_size, 
                            output_dim=embedding_dim, 
                            input_length=input_length))

    # --- 2. LSTM Layers (Tax et al. Structure) ---
    # Layer 1: Returns sequences to feed into the next LSTM layer
    model.add(LSTM(100, return_sequences=True, dropout=0.2))
    
    # Layer 2: Returns only the last hidden state
    model.add(LSTM(100, return_sequences=False, dropout=0.2))
    
    # --- 3. Output Layer ---
    model.add(Dense(vocab_size, activation='softmax'))
    
    model.compile(loss='categorical_crossentropy', 
                  optimizer='adam', 
                  metrics=['accuracy'])
    
    return model

In [ ]:
# ... [Your existing imports and functions: import_xes, clean_dataset, etc.] ...

# 1. Load and Process Data (Recommendation: use sliding_window=True for LSTM)
df_2013 = import_xes("datasets/BPI_Challenge_2013_incidents.xes")

# Define prefix lengths
GLOBAL_prefix_lengths = [5, 10, 15, 20] # Shortened for testing, use your full list

# WARNING: I recommend changing False to True inside your process_dataset wrapper
# or calling build_prefix_df manually with sliding_window=True
train_df, test_df, _, _ = process_dataset(df_2013, GLOBAL_prefix_lengths)

# 2. Prepare Tensors
processor = LSTMDataProcessor()
processor.fit(train_df, test_df)

X_train, y_train, _ = processor.transform(train_df)
X_test, y_test, y_test_indices = processor.transform(test_df)

print(f"Training shape: {X_train.shape}") # Should be (Num_Samples, Max_Len)

# 3. Build Model (Experimenting with Learned Embeddings)
model = build_tax_model(
    vocab_size=processor.vocab_size, 
    input_length=processor.max_len,
    embedding_dim=64,       # Dimension of the vector space
    embedding_type='learned' # Change to 'one_hot' to replicate strict Tax et al.
)

model.summary()

# 4. Train
# Use EarlyStopping to prevent overfitting
callback = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5)

history = model.fit(
    X_train, y_train,
    epochs=50,
    batch_size=128,
    validation_split=0.2,
    callbacks=[callback],
    verbose=1
)

# 5. Evaluate
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy: {accuracy:.4f}")

parsing log, completed traces :: 100%|██████████| 7554/7554 [00:02<00:00, 2923.17it/s]


Vocab Size: 5
Max Sequence Length: 20
Training shape: (176593, 20)


/Users/gijskoppenberg/Documents/UU Jaar 4/OZP/Resource-Centric-NAP/.venv/lib/python3.12/site-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_6 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
